<a href="https://colab.research.google.com/github/vishaljoshi24/DungeonsAndDragonsTurnClassification/blob/main/turn_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone 'https://github.com/vishaljoshi24/Dungeons-and-Dragons-Turn-Classification/'

In [ ]:
# !pip install dspy==3.1.0
!pip install dspy==3.2.1

In [ ]:
import pandas as pd
import dspy

# Data

In [ ]:
optimisation_df = pd.read_excel('optimisation_set_articulation.xlsx')
validation_df = pd.read_excel('validation_set_articulation.xlsx')

In [ ]:
optimisation_df

In [ ]:
optimisation_df.drop(columns=['Jack\'s Codes'], inplace=True)

In [ ]:
optimisation_df.drop(columns=['Agreed Codes'], inplace=True)

In [ ]:
optimisation_df.drop(columns=['Notes'], inplace=True)

In [ ]:
# optimisation_df.drop(columns=['Column1'], inplace=True)

In [ ]:
validation_df.drop(columns=['Jack\'s Codes'], inplace=True)

In [ ]:
validation_df.drop(columns=['Agreed Codes'], inplace=True)

In [ ]:
validation_df.drop(columns=['Notes'], inplace=True)

In [ ]:
optimisation_df

In [ ]:
optimisation_set = []

for context, question, response in optimisation_df.values:
    examples = dspy.Example(context=context, question=question, response=response).with_inputs("context", "question")
    optimisation_set.append(examples)

In [ ]:
optimisation_set

# Language Model

In [ ]:
lm = dspy.LM('ollama_chat/ministral-3', api_base = 'http://localhost:11434', api_key='', max_tokens=4096)
dspy.configure(lm=lm)

# Signatures

In [ ]:
from typing import Literal

class TurnClassifier(dspy.Signature):
    """Given the context for a Dungeons & Dragons game turn and the game turn itself, classify the turn."""
    context: str = dspy.InputField(desc = "The three previous game turns which describe a player's action or their dialogue.")
    question: str = dspy.InputField (desc="The current turn taken by a player, which can include a description of an action or a piece of dialogue.")
    response: Literal['assert game-state',
                      'question game-state',
                      'act on game-state',
                      'propose intention or action',
                      'probe intention',
                      'acknowledge',
                      'argue',
                      'interact with resource',
                      ] = dspy.OutputField()

class PlayerInstruction(dspy.Signature):
  category: Literal['assert game-state',
                      'question game-state',
                      'act on game-state',
                      'propose intention or action',
                      'probe intention',
                      'acknowledge',
                      'argue',
                      'interact with resource',
                    ] = dspy.InputField()
  player_instruction:str = dspy.OutputField(desc="instruction on how to behave within a D&D game.")


# Modules

In [ ]:
class ClassifyTurns(dspy.Module):
  def __init__(self):
    self.classifier = dspy.ChainOfThought(TurnClassifier, caching=False)

  def forward(self, context, question, **kwargs):
    prediction = self.classifier(context=context, question=question)
    return prediction

In [ ]:
classify = ClassifyTurns()
def classify_turn(context, turn):
    try:
        predicted_category = classify(context=context, question=question)
        return predicted_category
    except Exception as e:
        return 0

In [ ]:
validation_set = []

for context, question, response in validation_df.values:
    examples = dspy.Example(context=context, question=question, response=response).with_inputs("context", "question")
    validation_set.append(examples)

In [ ]:
true_categories = []

for i in range(len(validation_set)):
  true_categories.append(validation_set[i]['response'])

In [ ]:
true_categories

In [ ]:
label_list = [
    'knowledge request',
    'knowledge update',
    'knowledge share',
    'knowledge confirmation',
    'argumentation',
    'resource use',
    'resource share',
    'resource aid',
    'resource request',
    'enact narration',
    'deterministic action'
]

In [ ]:
# unoptimised_predictions = []

# for i in range(len(validation_set)):
#     unoptimised_predictions.append(classify_turn(validation_set[i]['context'], validation_set[i]['question']))

In [ ]:
# unoptimised_predicted_categories = []

# for i in range(len(unoptimised_predictions)):
#   unoptimised_predicted_categories.append(unoptimised_predictions[i].response)

In [ ]:
# from sklearn.metrics import f1_score

# def f1(true_categories, predicted_categories, trace=None):
#   for i in range(len(true_categories)):
#     return f1_score(true_categories, predicted_categories, average='weighted')

In [ ]:
# f1(true_categories, unoptimised_predicted_categories)

### Optimisation


In [ ]:
semanticF1 = dspy.evaluate.SemanticF1(threshold=0.66, decompositional=False)

In [ ]:
optimisation_set[0]

In [ ]:
optimizer_copro = dspy.COPRO(metric=semanticF1, breadth=10, depth=3, init_temperature=1.4, track_stats=True)
optimizer_fewshot = dspy.BootstrapFewShot(metric=semanticF1, metric_threshold=0.66, teacher_settings=None, max_bootstrapped_demos=4, max_labeled_demos=16)
optimizer_mipro = dspy.MIPROv2(metric=semanticF1, auto='light')

In [ ]:
!pip install dspy[optuna]

In [ ]:
import litellm

In [ ]:
litellm.drop_params=True

In [ ]:
fewshot_classifier = optimizer_fewshot.compile(classify, trainset=optimisation_set)

In [ ]:
fewshot_classifier.save("fewshot_classifier.json")

Evaluation

In [ ]:
# evaluate = dspy.Evaluate(devset=validation_set, metric = semanticF1, num_threads=1, provide_traceback=True)

In [ ]:
# evaluate(fewshot_classifier)

In [ ]:
predictions = []

for i in range(len(validation_set)):
    predictions.append(fewshot_classifier(validation_set[i]['context'], validation_set[i]['question']))

In [ ]:
predicted_categories = []

for i in range(len(predictions)):
  predicted_categories.append(predictions[i].response)

In [ ]:
true_categories = []

for i in range(len(validation_set)):
  true_categories.append(validation_set[i]['response'])

In [ ]:
from sklearn.metrics import f1_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score


def f1(true_categories, predicted_categories, trace=None):
  for i in range(len(true_categories)):
    return f1_score(true_categories, predicted_categories, average='weighted')

def precision(true_categories, predicted_categories, trace=None):
  for i in range(len(true_categories)):
    return precision_score(true_categories, predicted_categories, labels=label_list, average=None, zero_division=0)

def recall(true_categories, predicted_categories, trace=None):
  for i in range(len(true_categories)):
    return recall_score(true_categories, predicted_categories, labels = label_list, average=None, zero_division=0)

In [ ]:
recall_list = []
precision_list = []

for i in range(len(true_categories)):
  precision_list.append(precision(true_categories, predicted_categories))

for i in range(len(true_categories)):
  recall_list.append(recall(true_categories, predicted_categories))

In [ ]:
f1(true_categories, predicted_categories)